In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, Input
from tensorflow.keras.callbacks import EarlyStopping

# 1. Load Data
train_df = pd.read_csv('/kaggle/input/datasets/akulbansal25117021/cleaned2-0/cleaned_binary_train_70.csv')
val_df = pd.read_csv('/kaggle/input/datasets/akulbansal25117021/cleaned2-0/cleaned_binary_val_30.csv')
test_df = pd.read_csv('/kaggle/input/datasets/parambratasinha/gps-spoofing-detection-on-uav/NyneOS_test.csv')

# 2. Preprocessing
# Dropping 'channel' as it's categorical (string). Keep 'time' if you want, 
# but usually features start from PRN onwards.
drop_cols = ['channel', 'Output']
features = [c for c in train_df.columns if c not in drop_cols]

X_train_raw = train_df[features].values
y_train = train_df['Output'].values

X_val_raw = val_df[features].values
y_val = val_df['Output'].values

X_test_raw = test_df[[c for c in features if c in test_df.columns]].values

# Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_val_scaled = scaler.transform(X_val_raw)
X_test_scaled = scaler.transform(X_test_raw)

# Reshape for LSTM: (samples, time_steps, features)
# Treating each row as a single time step for this sequence
X_train = X_train_scaled.reshape((X_train_scaled.shape[0], 1, X_train_scaled.shape[1]))
X_val = X_val_scaled.reshape((X_val_scaled.shape[0], 1, X_val_scaled.shape[1]))
X_test = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))

# 3. Build Bidirectional LSTM Model
model = Sequential([
    Input(shape=(X_train.shape[1], X_train.shape[2])),
    Bidirectional(LSTM(64, return_sequences=True)),
    Dropout(0.3),
    Bidirectional(LSTM(32)),
    Dropout(0.3),
    Dense(16, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 4. Regularization: Early Stopping
early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=5, 
    restore_best_weights=True
)

# 5. Train
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=64,
    callbacks=[early_stop]
)

# We compute this at the row level first to see model performance
val_probs = model.predict(X_val).flatten()
val_preds = (val_probs > 0.5).astype(int)
f1 = f1_score(y_val, val_preds)
print(f"\nRow-level Validation F1 Score: {f1:.4f}")

# 6. Prediction on Test Set & Submission Formatting
# Get raw probabilities (confidence)
test_probs = model.predict(X_test).flatten()

# Create a temporary dataframe to group by timestamp
results_df = pd.DataFrame({
    'time': test_df['time'],
    'confidence': test_probs
})

# Aggregation Logic: 
# spoofed = 1 if any channel in that timestamp is > 0.5
# confidence = the maximum probability found among the channels for that timestamp
submission = results_df.groupby('time')['confidence'].agg(['max']).reset_index()
submission.columns = ['time', 'confidence']
submission['spoofed'] = (submission['confidence'] > 0.5).astype(int)

# Reorder columns to match requested format: time, spoofed, confidence
submission = submission[['time', 'spoofed', 'confidence']]

# 7. Save to CSV
submission.to_csv('submission.csv', index=False)
print("\nFinal submission file 'submission.csv' generated successfully.")
print(submission.head())

2026-04-17 02:56:56.445885: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776394616.655602      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776394616.712301      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776394617.184036      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776394617.184085      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776394617.184088      23 computation_placer.cc:177] computation placer alr

Epoch 1/50


I0000 00:00:1776394648.185017      68 cuda_dnn.cc:529] Loaded cuDNN version 91002


13841/13841 ━━━━━━━━━━━━━━━━━━━━ 143s 10ms/step - accuracy: 0.8870 - loss: 0.2667 - val_accuracy: 0.9632 - val_loss: 0.0949
Epoch 2/50
13841/13841 ━━━━━━━━━━━━━━━━━━━━ 135s 10ms/step - accuracy: 0.9569 - loss: 0.1048 - val_accuracy: 0.9769 - val_loss: 0.0610
Epoch 3/50
13841/13841 ━━━━━━━━━━━━━━━━━━━━ 134s 10ms/step - accuracy: 0.9669 - loss: 0.0820 - val_accuracy: 0.9722 - val_loss: 0.0680
Epoch 4/50
13841/13841 ━━━━━━━━━━━━━━━━━━━━ 134s 10ms/step - accuracy: 0.9714 - loss: 0.0709 - val_accuracy: 0.9801 - val_loss: 0.0463
Epoch 5/50
13841/13841 ━━━━━━━━━━━━━━━━━━━━ 135s 10ms/step - accuracy: 0.9752 - loss: 0.0620 - val_accuracy: 0.9841 - val_loss: 0.0380
Epoch 6/50
13841/13841 ━━━━━━━━━━━━━━━━━━━━ 136s 10ms/step - accuracy: 0.9772 - loss: 0.0563 - val_accuracy: 0.9867 - val_loss: 0.0331
Epoch 7/50
13841/13841 ━━━━━━━━━━━━━━━━━━━━ 141s 10ms/step - accuracy: 0.9787 - loss: 0.0529 - val_accuracy: 0.9884 - val_loss: 0.0286
Epoch 8/50
13841/13841 ━━━━━━━━━━━━━━━━━━━━ 137s 10ms/step - accur